<a href="https://colab.research.google.com/github/HB0918/NLPforET/blob/main/Multimodal_Comprehension_Diagnostic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio edge-tts nest_asyncio matplotlib -q

import gradio as gr
import random
import time
import asyncio
import edge_tts
import nest_asyncio
import os
import uuid
import matplotlib.pyplot as plt

nest_asyncio.apply()

# =====================================================
# PASSAGES
# =====================================================

passages = [
    {
        "title": "Families Cut Back on Fun Days Out Due to Rising Costs",
        "text": """
Many middle-income families in the UK are stopping their fun days out. Bianca and Paul Osborne earn an average salary, but they say everything is getting too expensive. They cannot justify spending over £120 on a simple lunch and a short activity for their two daughters. This is part of a growing trend where families cut back on luxuries because they have less money left after paying their monthly bills.

The main reason for this problem is high inflation. The cost of living is rising faster than people's wages. Because everyday items cost more, families feel a lot of financial pressure. Another family spent over £200 on dinner and bowling. They said it felt like spending their whole weekly grocery budget in just one night.

This situation also hurts local businesses. Cafes, restaurants, and play centers are losing customers. At the same time, their costs for rent and staff are going up. Some business owners even have to reduce their staff to survive. Because of these difficult times, families like the Osbornes now prioritize free activities. Instead of going to expensive restaurants or amusement parks, they choose to visit local parks and museums to make happy memories without spending money.
""",
        "questions": [
            {
                "question": "Why are middle-income families stopping their fun days out?",
                "options": [
                    "They do not enjoy going to restaurants and parks anymore.",
                    "They have less money because the cost of living is rising.",
                    "They want to save money to buy a new house."
                ],
                "answer": "They have less money because the cost of living is rising."
            },
            {
                "question": "How does the current economic situation affect local businesses?",
                "options": [
                    "They are losing customers while their operating costs increase.",
                    "They are making more money because families spend more on weekends.",
                    "They are hiring more staff to deal with the high number of visitors."
                ],
                "answer": "They are losing customers while their operating costs increase."
            },
            {
                "question": "What do families like the Osbornes do instead of spending money on expensive activities?",
                "options": [
                    "They stay at home and watch television all day.",
                    "They borrow money from the bank to go on holidays.",
                    "They choose to do free activities like visiting parks and museums."
                ],
                "answer": "They choose to do free activities like visiting parks and museums."
            }
        ]
    },

    {
        "title": "A New Car-Free Neighborhood in America",
        "text": """
In Tempe, Arizona, a new neighborhood called Culdesac is trying to reimagine city life. It is the first modern car-free neighborhood in the United States. Instead of wide roads for cars, it has narrow walking paths, beautiful white buildings, and open spaces. When people walk through the area, they feel like they are visiting a small town in Greece or Italy.

In the past, cities were built for cars, which caused pollution and made people feel lonely. However, modern urbanism is changing this. Culdesac focuses on making a sustainable environment. The white buildings keep the area cool, and the lack of cars reduces pollution. Because there are no cars, residents can easily meet their neighbors, which builds a strong community.

Although residents do not own cars, they still have excellent mobility. The neighborhood is close to public trains, and people can rent electric bikes or shared cars if they need to travel far. Culdesac shows that people can live happily and easily without owning a car.
""",
        "questions": [
            {
                "question": "Why is Culdesac considered unique in the United States?",
                "options": [
                    "It is the largest city in Arizona.",
                    "It is the first modern car-free neighborhood in the U.S.",
                    "It is built only for tourists from Europe."
                ],
                "answer": "It is the first modern car-free neighborhood in the U.S."
            },
            {
                "question": "How does the neighborhood help reduce pollution?",
                "options": [
                    "Residents use larger cars with cleaner fuel.",
                    "The area has more highways and parking spaces.",
                    "The lack of cars and the design of the buildings support sustainability."
                ],
                "answer": "The lack of cars and the design of the buildings support sustainability."
            },
            {
                "question": "How can residents travel long distances if they do not own cars?",
                "options": [
                    "They can use public transportation or shared vehicles.",
                    "They must walk everywhere.",
                    "They can only travel by airplane."
                ],
                "answer": "They can use public transportation or shared vehicles."
            }
        ]
    },

    {
        "title": "Football's Fake Image Problem: How AI is Changing the Game",
        "text": """
Today, you can easily find fake images and videos of famous football players on social media. Because of artificial intelligence (AI), people can make pictures of players doing strange things, like serving burgers or wearing historical clothes. Many of these images seem like harmless fun. However, it is becoming very difficult to tell what is real and what is fake.

As football is a huge business, players and clubs must protect their image. They want to control their brand and stop unauthorized people from using their names. If a fake video shows a player doing something bad, it could damage their reputation. Because of this, clubs are looking for ways to stop these fake images.

However, it is hard to enforce the law against AI creators. Going to court takes a long time and costs a lot of money. Instead, experts say it is faster to challenge the social media platforms directly. New laws may require these platforms to remove illegal content. In the future, social media apps might also force users to add an "AI generated" label to their videos.
""",
        "questions": [
            {
                "question": "Why are fake AI-generated football images becoming a problem?",
                "options": [
                    "They make football games more expensive.",
                    "It is becoming difficult to distinguish real content from fake content.",
                    "Players enjoy creating fake videos for entertainment."
                ],
                "answer": "It is becoming difficult to distinguish real content from fake content."
            },
            {
                "question": "Why are football clubs worried about fake videos?",
                "options": [
                    "Fake videos can damage players’ and clubs’ reputations.",
                    "Fake videos improve the popularity of football teams.",
                    "Football clubs want to create more AI-generated content."
                ],
                "answer": "Fake videos can damage players’ and clubs’ reputations."
            },
            {
                "question": "What solution do experts suggest for dealing with fake AI content?",
                "options": [
                    "Closing all social media platforms permanently.",
                    "Preventing people from using the internet.",
                    "Requiring platforms to remove illegal content and label AI-generated videos."
                ],
                "answer": "Requiring platforms to remove illegal content and label AI-generated videos."
            }
        ]
    }
]

# =====================================================
# RANDOM MODALITY ASSIGNMENT
# =====================================================

modes = ["LO", "RO", "RWL"]
random.shuffle(modes)

for i in range(3):
    passages[i]["mode"] = modes[i]

# =====================================================
# GENERATE TTS
# =====================================================

async def generate_tts(text, filename):
    communicate = edge_tts.Communicate(text, voice="en-US-JennyNeural")
    await communicate.save(filename)

for i, passage in enumerate(passages):

    filename = f"audio_{i}.mp3"

    if not os.path.exists(filename):
        asyncio.run(generate_tts(passage["text"], filename))

    passage["audio"] = filename

# =====================================================
# STORAGE
# =====================================================

results = {}
start_times = {}

# =====================================================
# TIMER
# =====================================================

def start_timer(session_name):
    start_times[session_name] = time.time()

    return """
    <div style='font-size:28px; color:green; font-weight:bold;'>
    Session Started!
    </div>
    """

# =====================================================
# CHECK ANSWERS
# =====================================================

def check_answers(session_name, passage, a1, a2, a3):

    elapsed = round(time.time() - start_times[session_name], 2)

    answers = [a1, a2, a3]

    correct = 0

    for i, ans in enumerate(answers):
        if ans == passage["questions"][i]["answer"]:
            correct += 1

    score = round(correct / 3 * 100, 1)

    results[passage["mode"]] = {
        "score": score,
        "time": elapsed
    }

    return f"""
    <div style='font-size:30px; line-height:2; font-weight:bold;'>

    ✅ Accuracy: {score}%<br>

    ⏱ Study Time: {elapsed} seconds

    </div>
    """

# =====================================================
# FINAL RESULTS
# =====================================================

def make_chart():

    if len(results) < 3:
        return None, """
        <div style='font-size:30px; color:red; font-weight:bold;'>
        Please complete all sessions first.
        </div>
        """

    labels = ["LO", "RO", "RWL"]

    scores = [results[m]["score"] for m in labels]
    times = [results[m]["time"] for m in labels]

    fig, ax = plt.subplots(figsize=(7,5))

    ax.bar(labels, scores)

    ax.set_ylim(0,100)
    ax.set_ylabel("Accuracy (%)", fontsize=14)
    ax.set_title("Comprehension by Modality", fontsize=16)

    filename = f"{uuid.uuid4().hex}.png"

    plt.savefig(filename, bbox_inches="tight")
    plt.close()

    best = labels[scores.index(max(scores))]

    summary = f"""
    <div style='font-size:30px; line-height:2;'>

    🏆 <b>Best Mode:</b> {best}

    <br><br>

    🎧 <b>LO</b><br>
    Accuracy: {results['LO']['score']}%<br>
    Study Time: {results['LO']['time']} sec

    <br><br>

    📖 <b>RO</b><br>
    Accuracy: {results['RO']['score']}%<br>
    Study Time: {results['RO']['time']} sec

    <br><br>

    🎧📖 <b>RWL</b><br>
    Accuracy: {results['RWL']['score']}%<br>
    Study Time: {results['RWL']['time']} sec

    <br><br>

    💡 Higher scores indicate stronger comprehension.<br>
    ⏱ Longer study time may indicate heavier processing load.

    </div>
    """

    return filename, summary

# =====================================================
# RESTUDY
# =====================================================

def load_restudy(title, mode):

    selected = None

    for p in passages:
        if p["title"] == title:
            selected = p
            break

    if selected is None:
        return "", None

    if mode == "LO":
        return "🎧 Listening Only Mode", selected["audio"]

    elif mode == "RO":
        return selected["text"], None

    elif mode == "RWL":
        return selected["text"], selected["audio"]

# =====================================================
# UI
# =====================================================

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🎧📖 Multimodal Comprehension Diagnostic

    Discover which learning condition helps your comprehension the most.

    - LO = Listening Only
    - RO = Reading Only
    - RWL = Reading While Listening

    Complete all 3 sessions and compare your performance.
    """)

    for idx, passage in enumerate(passages):

        session_name = f"session_{idx}"

        with gr.Tab(f"Session {idx+1} - {passage['mode']}"):

            gr.Markdown(f"## {passage['title']}")

            start_btn = gr.Button("▶ Start Session")

            timer_output = gr.HTML()

            start_btn.click(
                fn=lambda s=session_name: start_timer(s),
                outputs=timer_output
            )

            if passage["mode"] == "LO":

                gr.Markdown("## 🎧 Listening Only")

                gr.Audio(
                    value=passage["audio"],
                    autoplay=False
                )

            elif passage["mode"] == "RO":

                gr.Markdown("## 📖 Reading Only")

                gr.Textbox(
                    value=passage["text"],
                    lines=18,
                    interactive=False
                )

            elif passage["mode"] == "RWL":

                gr.Markdown("## 🎧📖 Reading While Listening")

                gr.Textbox(
                    value=passage["text"],
                    lines=18,
                    interactive=False
                )

                gr.Audio(
                    value=passage["audio"],
                    autoplay=False
                )

            gr.Markdown("## 📝 Comprehension Quiz")

            q1 = gr.Radio(
                choices=passage["questions"][0]["options"],
                label=passage["questions"][0]["question"]
            )

            q2 = gr.Radio(
                choices=passage["questions"][1]["options"],
                label=passage["questions"][1]["question"]
            )

            q3 = gr.Radio(
                choices=passage["questions"][2]["options"],
                label=passage["questions"][2]["question"]
            )

            submit_btn = gr.Button("Submit Answers")

            result_box = gr.HTML()

            submit_btn.click(
                fn=lambda a1, a2, a3, p=passage, s=session_name:
                    check_answers(s, p, a1, a2, a3),
                inputs=[q1, q2, q3],
                outputs=result_box
            )

    # =====================================================
    # FINAL RESULTS TAB
    # =====================================================

    with gr.Tab("📊 Final Results"):

        gr.Markdown("# 📊 Final Performance Dashboard")

        result_btn = gr.Button("Generate Results")

        chart_output = gr.Image()

        text_output = gr.HTML()

        result_btn.click(
            fn=make_chart,
            outputs=[chart_output, text_output]
        )

    # =====================================================
    # RESTUDY TAB
    # =====================================================

    with gr.Tab("🔁 Restudy Mode"):

        gr.Markdown("""
        # 🔁 Study the Same Passage in Another Mode

        After checking your results, you can restudy the same passage
        using a different modality.
        """)

        passage_dropdown = gr.Dropdown(
            choices=[p["title"] for p in passages],
            label="Choose Passage"
        )

        mode_dropdown = gr.Dropdown(
            choices=["LO", "RO", "RWL"],
            label="Choose New Learning Mode"
        )

        load_btn = gr.Button("Load Study Mode")

        restudy_text = gr.Textbox(
            lines=18,
            label="Text"
        )

        restudy_audio = gr.Audio(
            label="Audio"
        )

        load_btn.click(
            fn=load_restudy,
            inputs=[passage_dropdown, mode_dropdown],
            outputs=[restudy_text, restudy_audio]
        )

demo.launch(debug=True)

/tmp/ipykernel_4576/1374831070.py:316: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7f559b18bc8757e8eb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
